In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(
    "../data/ecommerce_dataset__1m.csv",
    nrows=50000
)

print("✓ CSV File loaded successfully")
print(f"Rows loaded  : {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print(df.head(2).to_string())


✓ CSV File loaded successfully
Rows loaded  : 50,000
Total columns: 62
    order_id                  order_date  order_year  order_month  order_day  order_hour  order_minute  order_second is_weekend order_status return_reason customer_id    customer_name gender  age customer_segment        country         city  customer_loyalty_score  total_orders_by_customer account_creation_date product_id            product_name category sub_category brand  product_rating_avg  product_reviews_count  stock_quantity  unit_price_usd  quantity  discount_percent  discount_amount_usd  total_price_usd  cost_usd  profit_usd  tax_usd currency payment_method payment_status installment_plan shipping_method  shipping_cost_usd  delivery_days shipping_country warehouse_location delivery_status  rating review_sentiment     customer_feedback coupon_used coupon_code campaign_source device_type traffic_source  session_duration_minutes  pages_visited abandoned_cart_before  fraud_risk_score  profit_margin_percent order

In [2]:
# Show every column name and its data type
# This tells us which columns are text (object), numbers (int64/float64),
# or dates (datetime64)

print("All columns and data types:")
print("=" * 50)
for i, (col, dtype) in enumerate(df.dtypes.items(), 1):
    print(f"  {i:2}. {col:<40} {str(dtype)}")

All columns and data types:
   1. order_id                                 object
   2. order_date                               object
   3. order_year                               int64
   4. order_month                              int64
   5. order_day                                int64
   6. order_hour                               int64
   7. order_minute                             int64
   8. order_second                             int64
   9. is_weekend                               object
  10. order_status                             object
  11. return_reason                            object
  12. customer_id                              object
  13. customer_name                            object
  14. gender                                   object
  15. age                                      int64
  16. customer_segment                         object
  17. country                                  object
  18. city                                     object
  19. c

In [ ]:
missing_count = df.isnull().sum()
missing_pct   = (missing_count / len(df) * 100).round(2)

# Build a clean summary table showing only columns that HAVE missing values
missing_df = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %'    : missing_pct
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if len(missing_df) == 0:
    print("No missing values found.")
else:
    print(f"Columns with missing values ({len(missing_df)} found):")
    print("=" * 45)
    print(missing_df.to_string())
    print("\nExplanation of each missing column:")
    print("  return_reason  → NULL when order was NOT returned (correct behaviour)")
    print("  coupon_code    → NULL when no coupon was used (correct behaviour)")
    print("  customer_feedback → NULL when customer left no feedback (normal)")

Columns with missing values (3 found):
                   Missing Count  Missing %
return_reason              45018      90.04
coupon_code                24838      49.68
customer_feedback          10067      20.13

Explanation of each missing column:
  return_reason  → NULL when order was NOT returned (correct behaviour)
  coupon_code    → NULL when no coupon was used (correct behaviour)
  customer_feedback → NULL when customer left no feedback (normal)


In [ ]:
categorical_cols = [
    'order_status', 'is_weekend', 'return_reason',
    'customer_segment', 'gender',
    'country', 'warehouse_location',
    'category', 'sub_category', 'brand',
    'shipping_method', 'delivery_status',
    'payment_method', 'payment_status', 'installment_plan',
    'order_priority', 'support_ticket_created', 'coupon_used',
    'campaign_source', 'device_type', 'traffic_source'
]

print("CATEGORICAL COLUMNS — UNIQUE VALUES")
print("=" * 60)
for col in categorical_cols:
    unique_vals = df[col].dropna().unique().tolist()
    n = df[col].nunique()
    counts = df[col].value_counts()
    print(f"\n{col}  ({n} unique values)")
    print(f"  Values: {unique_vals}")
    print(f"  Counts:")
    for val, cnt in counts.items():
        pct = cnt / len(df) * 100
        print(f"    {str(val):<30} {cnt:>6,}  ({pct:.1f}%)")

CATEGORICAL COLUMNS — UNIQUE VALUES

order_status  (5 unique values)
  Values: ['Completed', 'Returned', 'Pending', 'Cancelled', 'Processing']
  Counts:
    Completed                      34,940  (69.9%)
    Pending                         5,003  (10.0%)
    Returned                        4,982  (10.0%)
    Cancelled                       2,574  (5.1%)
    Processing                      2,501  (5.0%)

is_weekend  (2 unique values)
  Values: ['No', 'Yes']
  Counts:
    No                             35,571  (71.1%)
    Yes                            14,429  (28.9%)

return_reason  (5 unique values)
  Values: ['Not As Described', 'Better Price Elsewhere', 'Defective Product', 'Changed Mind', 'Wrong Item']
  Counts:
    Defective Product               1,013  (2.0%)
    Better Price Elsewhere          1,009  (2.0%)
    Changed Mind                    1,001  (2.0%)
    Wrong Item                        989  (2.0%)
    Not As Described                  970  (1.9%)

customer_segment  (3 uni

In [ ]:
measure_cols = [
    'unit_price_usd', 'quantity', 'discount_percent',
    'total_price_usd', 'cost_usd', 'profit_usd',
    'tax_usd', 'profit_margin_percent', 'shipping_cost_usd',
    'delivery_days', 'age', 'customer_loyalty_score'
]

print("NUMERIC MEASURES — STATISTICAL SUMMARY")
print("=" * 70)
summary = df[measure_cols].describe().round(2)
print(summary.to_string())

# Also show total sums for the additive measures
print("\n\nTOTAL SUMS (additive measures only — on 50k sample)")
print("=" * 50)
additive = ['quantity', 'total_price_usd', 'cost_usd',
            'profit_usd', 'tax_usd', 'shipping_cost_usd']
for col in additive:
    total = df[col].sum()
    avg   = df[col].mean()
    print(f"  {col:<25} Total: {total:>15,.2f}   Avg per order: {avg:>8,.2f}")

# Check if profit = revenue - cost (data consistency check)
print("\n\nDATA CONSISTENCY CHECK")
print("=" * 50)
df['calc_profit'] = df['total_price_usd'] - df['cost_usd']
diff = (df['profit_usd'] - df['calc_profit']).abs()
print(f"  Max difference (profit_usd vs total-cost): {diff.max():.4f}")
print(f"  → If close to 0, profit column is consistent ✓")
df.drop(columns=['calc_profit'], inplace=True)

NUMERIC MEASURES — STATISTICAL SUMMARY
       unit_price_usd  quantity  discount_percent  total_price_usd  cost_usd  profit_usd   tax_usd  profit_margin_percent  shipping_cost_usd  delivery_days       age  customer_loyalty_score
count        50000.00  50000.00          50000.00         50000.00  50000.00    50000.00  50000.00               50000.00           50000.00       50000.00  50000.00                50000.00
mean           147.22      3.00              8.49           403.36    242.23      161.13     30.27                  39.48              12.55           7.55     46.37                   50.04
std            103.71      1.42              7.60           369.69    226.55      161.68     28.85                  10.94               7.21           4.03     16.74                   28.94
min             10.02      1.00              0.00             8.33      4.30        0.82      0.46                   6.67               0.00           1.00     18.00                    0.00
25%        

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])

print("DATE DIMENSION ANALYSIS")
print("=" * 50)
print(f"  Earliest order : {df['order_date'].min().date()}")
print(f"  Latest order   : {df['order_date'].max().date()}")
print(f"  Total date span: {(df['order_date'].max() - df['order_date'].min()).days} days")

print(f"\nYears in data:")
year_counts = df['order_year'].value_counts().sort_index()
for yr, cnt in year_counts.items():
    print(f"  {yr}: {cnt:,} orders")

print(f"\nMonths (all 12 present?):")
print(f"  Unique months: {sorted(df['order_month'].unique())}")

print(f"\nWeekend vs Weekday split:")
wkend = df['is_weekend'].value_counts()
for k, v in wkend.items():
    print(f"  {k}: {v:,} ({v/len(df)*100:.1f}%)")

print(f"\nHierarchy levels available:")
print(f"  Level 1: order_day     (1–31)")
print(f"  Level 2: order_month   (1–12)")
print(f"  Level 3: quarter       (derived: month // 3)")
print(f"  Level 4: order_year    ({df['order_year'].min()}–{df['order_year'].max()})")
print(f"  Extra  : is_weekend    (True/False flag)")

DATE DIMENSION ANALYSIS
  Earliest order : 2024-02-03
  Latest order   : 2026-02-02
  Total date span: 730 days

Years in data:
  2024: 22,806 orders
  2025: 24,940 orders
  2026: 2,254 orders

Months (all 12 present?):
  Unique months: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]

Weekend vs Weekday split:
  No: 35,571 (71.1%)
  Yes: 14,429 (28.9%)

Hierarchy levels available:
  Level 1: order_day     (1–31)
  Level 2: order_month   (1–12)
  Level 3: quarter       (derived: month // 3)
  Level 4: order_year    (2024–2026)
  Extra  : is_weekend    (True/False flag)


In [ ]:
total_rows    = len(df)
unique_orders = df['order_id'].nunique()

print("GRAIN VERIFICATION")
print("=" * 50)
print(f"  Total rows in sample  : {total_rows:,}")
print(f"  Unique order_id values: {unique_orders:,}")
print()

if total_rows == unique_orders:
    print("  ✓ GRAIN CONFIRMED: Every row has a unique order_id.")
    print("    Grain = one row = one unique sales order.")
else:
    duplicates = total_rows - unique_orders
    print(f"  ✗ WARNING: {duplicates:,} duplicate order_ids found.")
    print("    Investigate before proceeding.")
    dupes = df[df.duplicated(subset='order_id', keep=False)]
    print(dupes[['order_id','order_date','customer_id']].head(10))

# Also check customer_id cardinality
print(f"\n  Unique customers in sample : {df['customer_id'].nunique():,}")
print(f"  Unique products in sample  : {df['product_id'].nunique():,}")
print(f"  → Each order has exactly 1 customer and 1 product ✓")

GRAIN VERIFICATION
  Total rows in sample  : 50,000
  Unique order_id values: 49,979

  ✗ WARNING: 21 duplicate order_ids found.
    Investigate before proceeding.
       order_id                 order_date customer_id
203   ORD-OM5FK 2024-12-15 23:17:20.504002   CUS-ISWLW
301   ORD-87WS1 2024-02-14 01:13:46.466891   CUS-OQSOT
613   ORD-SJ64Y 2024-08-24 15:04:06.165913   CUS-5R1F5
1262  ORD-F1HRJ 2025-10-23 21:31:02.949821   CUS-33VVO
2594  ORD-Q3MH4 2025-07-15 17:05:24.161925   CUS-EDFZE
2778  ORD-GLLEG 2025-05-29 15:38:52.470394   CUS-46QFN
3339  ORD-D686O 2025-04-01 09:15:40.729545   CUS-J5PAL
7153  ORD-87WS1 2025-08-14 10:09:58.099389   CUS-8ASG3
8006  ORD-O9LQQ 2024-04-21 20:24:10.597497   CUS-FKZE9
9409  ORD-WC2I4 2025-01-28 11:59:12.646289   CUS-BN6X9

  Unique customers in sample : 49,973
  Unique products in sample  : 49,271
  → Each order has exactly 1 customer and 1 product ✓


In [ ]:
print("PRODUCT DIMENSION HIERARCHY")
print("=" * 60)

# Level 1: Categories
cats = df['category'].value_counts()
print(f"\nLevel 1 — Category ({df['category'].nunique()} values):")
for cat, cnt in cats.items():
    print(f"  {cat:<20} {cnt:,} orders")

print(f"\nLevel 2 — Sub-category ({df['sub_category'].nunique()} values):")
for cat in df['category'].unique():
    subcats = df[df['category']==cat]['sub_category'].unique()
    print(f"  {cat}:")
    for sc in sorted(subcats):
        print(f"    └─ {sc}")

print(f"\nBrand branch ({df['brand'].nunique()} brands):")
print(f"  {sorted(df['brand'].unique())}")

print(f"\nUnique products: {df['product_name'].nunique()}")
print(f"  Sample: {df['product_name'].unique()[:8].tolist()}")

print(f"\nProduct rating_avg range: {df['product_rating_avg'].min()} – {df['product_rating_avg'].max()}")

PRODUCT DIMENSION HIERARCHY

Level 1 — Category (5 values):
  Clothing             10,126 orders
  Electronics          10,004 orders
  Sports               10,003 orders
  Home                 9,957 orders
  Health               9,910 orders

Level 2 — Sub-category (22 values):
  Home:
    └─ Appliances
    └─ Bedding
    └─ Decor
    └─ Furniture
    └─ Kitchen
  Health:
    └─ Fitness
    └─ Medical Devices
    └─ Personal Care
    └─ Supplements
  Sports:
    └─ Accessories
    └─ Gym Equipment
    └─ Outdoor
    └─ Sports Wear
  Electronics:
    └─ Accessories
    └─ Audio
    └─ Cameras
    └─ Laptops
    └─ Smartphones
    └─ Tablets
  Clothing:
    └─ Accessories
    └─ Kids Wear
    └─ Mens Wear
    └─ Shoes
    └─ Womens Wear

Brand branch (20 brands):
  ['Adidas', 'Anker', 'Apple', 'Asus', 'Baseus', 'Canon', 'Dell', 'Fitbit', 'GoPro', 'HP', 'Huawei', 'JBL', 'Lenovo', 'Logitech', 'Nike', 'Nikon', 'Philips', 'Samsung', 'Sony', 'Xiaomi']

Unique products: 48
  Sample: ['Coffee 

In [ ]:
print("GEOGRAPHY DIMENSION")
print("=" * 60)

print(f"\nCountries ({df['country'].nunique()} total):")
for country, cnt in df['country'].value_counts().items():
    print(f"  {country:<20} {cnt:,} orders  ({cnt/len(df)*100:.1f}%)")

print(f"\nCities per country (sample — first 3 cities each):")
for country in sorted(df['country'].unique()):
    cities = df[df['country']==country]['city'].unique()
    print(f"  {country}: {list(cities[:3])} ...")

print(f"\nWarehouse locations ({df['warehouse_location'].nunique()} total):")
for wh, cnt in df['warehouse_location'].value_counts().items():
    print(f"  {wh:<20} {cnt:,} orders  ({cnt/len(df)*100:.1f}%)")

GEOGRAPHY DIMENSION

Countries (10 total):
  Netherlands          5,101 orders  (10.2%)
  Australia            5,060 orders  (10.1%)
  Spain                5,047 orders  (10.1%)
  Italy                5,030 orders  (10.1%)
  Canada               5,013 orders  (10.0%)
  Belgium              4,977 orders  (10.0%)
  United Kingdom       4,968 orders  (9.9%)
  Germany              4,951 orders  (9.9%)
  United States        4,950 orders  (9.9%)
  France               4,903 orders  (9.8%)

Cities per country (sample — first 3 cities each):
  Australia: ['Adelaide', 'Melbourne', 'Sydney'] ...
  Belgium: ['Bruges', 'Ghent', 'Antwerp'] ...
  Canada: ['Calgary', 'Toronto', 'Vancouver'] ...
  France: ['Nice', 'Paris', 'Lyon'] ...
  Germany: ['Berlin', 'Frankfurt', 'Munich'] ...
  Italy: ['Milan', 'Turin', 'Naples'] ...
  Netherlands: ['Utrecht', 'Amsterdam', 'The Hague'] ...
  Spain: ['Madrid', 'Valencia', 'Bilbao'] ...
  United Kingdom: ['London', 'Glasgow', 'Manchester'] ...
  United States: [

In [ ]:
print("ORDER DIMENSION — STATUS AND RETURN ANALYSIS")
print("=" * 60)

print("\nOrder status:")
for status, cnt in df['order_status'].value_counts().items():
    print(f"  {status:<15} {cnt:,}  ({cnt/len(df)*100:.1f}%)")

returned = df[df['order_status'] == 'Returned']
print(f"\nReturn reasons (applies to {len(returned):,} returned orders only):")
for reason, cnt in returned['return_reason'].value_counts().items():
    print(f"  {reason:<35} {cnt:,}  ({cnt/len(returned)*100:.1f}%)")

not_returned = df[df['order_status'] != 'Returned']
has_reason_but_not_returned = not_returned['return_reason'].notna().sum()
print(f"\nBusiness rule check:")
print(f"  Non-returned orders WITH a return_reason: {has_reason_but_not_returned}")
if has_reason_but_not_returned == 0:
    print(f"  ✓ Rule holds — return_reason is NULL for all non-returned orders")
else:
    print(f"  ✗ Data quality issue — investigate these rows")

print(f"\nSupport tickets by order status:")
ticket_matrix = df.groupby('order_status')['support_ticket_created'].value_counts().unstack(fill_value=0)
print(ticket_matrix.to_string())

print(f"\nOrder priority distribution:")
for p, cnt in df['order_priority'].value_counts().items():
    print(f"  {p:<10} {cnt:,}  ({cnt/len(df)*100:.1f}%)")

ORDER DIMENSION — STATUS AND RETURN ANALYSIS

Order status:
  Completed       34,940  (69.9%)
  Pending         5,003  (10.0%)
  Returned        4,982  (10.0%)
  Cancelled       2,574  (5.1%)
  Processing      2,501  (5.0%)

Return reasons (applies to 4,982 returned orders only):
  Defective Product                   1,013  (20.3%)
  Better Price Elsewhere              1,009  (20.3%)
  Changed Mind                        1,001  (20.1%)
  Wrong Item                          989  (19.9%)
  Not As Described                    970  (19.5%)

Business rule check:
  Non-returned orders WITH a return_reason: 0
  ✓ Rule holds — return_reason is NULL for all non-returned orders

Support tickets by order status:
support_ticket_created     No    Yes
order_status                        
Cancelled                1289   1285
Completed               17688  17252
Pending                  2494   2509
Processing               1210   1291
Returned                 2531   2451

Order priority distribution:
